# 🏆 Hệ thống Nhận diện Rác 2 Giai đoạn (Final Pipeline)

Notebook này thực thi **TOÀN BỘ** quy trình của đồ án từ đầu đến cuối trên Kaggle.

### 📐 Kiến trúc đã chốt:
1. **Stage 1 (Phát hiện vùng rác):** Dùng **YOLOv8s** gốc, đặt `conf=0.15` để quét và lấy bằng được mọi thứ nghi ngờ là rác (Tối đa hóa Recall).
2. **Stage 2 (Phân loại & Lọc):** Dùng **EfficientNet-B2** với 6 lớp (Thêm lớp `Background` để lọc các cú lừa từ Stage 1).

⚠️ **Yêu cầu hệ thống:** GPU (T4 x2 hoặc P100) + Bật Internet.

In [1]:
# ============================================================
# 1. Tải Mã nguồn & Cài đặt thư viện
# ============================================================
!git clone https://github.com/Shiba-dotcom/waste-detection2-Stage.git
!pip install -q ultralytics timm

Cloning into 'waste-detection2-Stage'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 152 (delta 8), reused 18 (delta 7), pack-reused 126 (from 1)
Receiving objects: 100% (152/152), 101.25 MiB | 30.42 MiB/s, done.
Resolving deltas: 100% (46/46), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 90.8 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba

In [2]:
# ============================================================
# 2. Nhập Dữ liệu Ngoại lai (TACO, TrashNet, RealWaste)
# ============================================================
import os, shutil

!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/TrashNet
!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/RealWaste
!mkdir -p /kaggle/working/waste-detection2-Stage/data/raw

datasets_to_copy = [
    {"src": "/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/TrashNet"},
    {"src": "/kaggle/input/datasets/sohamchaudhari2004/taco-trash-detection-dataset/data",
     "dst": "/kaggle/working/waste-detection2-Stage/data/raw"},
    {"src": "/kaggle/input/datasets/joebeachcapital/realwaste/realwaste-main/RealWaste",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/RealWaste"}
]

for task in datasets_to_copy:
    if os.path.exists(task["src"]):
        os.makedirs(task["dst"], exist_ok=True)
        shutil.copytree(task["src"], task["dst"], dirs_exist_ok=True)
        print(f"Đã tải: {os.path.basename(task['src'])}")
    else:
        print(f"Bỏ qua: {task['src']} (Không tìm thấy trên Kaggle Dataset)")

Đã tải: dataset-resized
Đã tải: data
Đã tải: RealWaste


In [3]:
# ============================================================
# 3. Chạy Toàn bộ Tiền xử lý Dữ liệu (Data Pipeline)
# ============================================================
%cd /kaggle/working/waste-detection2-Stage

print("\n--- 3.1 Dọn dẹp & Tạo nhãn YOLO ---")
!python src/data_prep/data_cleaning.py
!python src/Training_dataYolo.py
!python src/data_prep/split_dataset.py

print("\n--- 3.2 Chuẩn bị dữ liệu cho Stage 2 (Classifier) ---")
!python src/data_prep/crop_for_classification.py
!python src/data_prep/merge_external_datasets.py
!python src/data_prep/generate_background.py

print("\n✅ Hoàn tất chuẩn bị 100% dữ liệu!")

/kaggle/working/waste-detection2-Stage

--- 3.1 Dọn dẹp & Tạo nhãn YOLO ---
  DATA CLEANING - TACO Dataset

[Load] annotations.json ...
  So anh goc         : 1500
  So annotations goc : 4784
  So categories      : 60

────────────────────────────────────────────────────────────
[Buoc 1] Kiem tra dong trung lap (Duplicates)
────────────────────────────────────────────────────────────
  Duplicate annotations: 0
  Duplicate image IDs: 0
  Duplicate file names: 0

────────────────────────────────────────────────────────────
[Buoc 2] Kiem tra gia tri thieu (Missing Values)
────────────────────────────────────────────────────────────
  Images: Tat ca truong bat buoc day du
  Annotations: Tat ca truong bat buoc day du
  Anh khong co annotation: 0

────────────────────────────────────────────────────────────
[Buoc 3] Kiem tra nhan khong hop le
────────────────────────────────────────────────────────────
  Annotations voi category_id khong hop le: 0
  Categories khong co trong mapping.csv: 1
 

In [4]:
# ============================================================
# 4. Huấn luyện Stage 1 (YOLOv8s - Binary Detector)
# BỎ QUA CELL NÀY NẾU BẠN ĐÃ CÓ FILE stage1_best.pt
# ============================================================
from ultralytics import YOLO
import time
from pathlib import Path

DATASET_YAML = "/kaggle/working/waste-detection2-Stage/data/processed_binary/dataset.yaml"
OUTPUT_DIR   = "/kaggle/working/waste-detection2-Stage/results/yolo8s_runs"

# Kiểm tra xem đã có model train sẵn chưa để bỏ qua
if not Path("stage1_best.pt").exists():
    print("🚀 Bắt đầu train Stage 1 (YOLOv8s)")
    model_s = YOLO("yolov8s.pt")
    t0 = time.time()
    model_s.train(
        data=DATASET_YAML, imgsz=640, epochs=100, batch=16, patience=20,
        optimizer="auto", lr0=0.01, cos_lr=True, augment=True, workers=4, device=[0,1],
        project=OUTPUT_DIR, name="stage1", exist_ok=True, save=True, save_period=-1,
    )
    print(f"\n✅ Train Stage 1 hoàn tất sau {(time.time()-t0)/60:.1f} phút")
    
    # Copy model tốt nhất ra thư mục gốc để dễ dùng
    !cp results/yolo8s_runs/stage1/weights/best.pt stage1_best.pt
else:
    print("⏭️ Đã tìm thấy 'stage1_best.pt', bỏ qua bước huấn luyện Stage 1.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 Bắt đầu train Stage 1 (YOLOv8s)
Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/waste-detection2-Stage/data/processed_binary/dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, e

In [5]:
# ============================================================
# 5. Huấn luyện Stage 2 (EfficientNet-B2 - 6 Lớp Classifier)
# ============================================================
# Train với các cơ chế: Freeze/Unfreeze, Dropout 0.4, RandAugment, Class Weights

!python src/train_stage2_classifier.py

[INFO] PyTorch : 2.10.0+cu128
[INFO] timm    : 1.0.26
[INFO] CUDA    : True
[INFO] GPU     : Tesla T4
[INFO] Device: cuda
[INFO] Data  : /kaggle/working/waste-detection2-Stage/data/classification_merged
[INFO] Augmentation đã được tăng cường:
  + RandAugment(num_ops=2, magnitude=9)
  + RandomVerticalFlip(p=0.2)
  + RandomErasing(p=0.3)
  + ColorJitter mạnh hơn (brightness/contrast 0.4)
  + RandomRotation(30°)
  LOADING DATASET
  Lớp phát hiện : ['Glass', 'Metal', 'Other', 'Paper', 'Plastic']
Traceback (most recent call last):
  File "/kaggle/working/waste-detection2-Stage/src/train_stage2_classifier.py", line 223, in <module>
    assert actual_classes == CLASS_NAMES, (
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: [LỖI] Thứ tự lớp không khớp!
  Mong đợi : ['Background', 'Glass', 'Metal', 'Other', 'Paper', 'Plastic']
  Thực tế  : ['Glass', 'Metal', 'Other', 'Paper', 'Plastic']


In [6]:
# ============================================================
# 6. Đánh giá Toàn Trình (End-to-End 2-Stage Pipeline)
# ============================================================
# Bước này mô phỏng luồng chạy thực tế: Đưa ảnh vào YOLO -> Cắt rác -> Phân loại

!python src/evaluate_2stage.py \
  --detector stage1_best.pt \
  --classifier models/stage2_best.pth \
  --data-dir data/processed_binary/images/test \
  --label-dir data/processed/labels/test \
  --conf 0.15 \
  --output results/2stage_eval

print("\n✅ ĐÁNH GIÁ HOÀN TẤT! Kết quả được lưu tại results/2stage_eval")

[INFO] Đang sử dụng thiết bị: cuda
[WARN] Không tìm thấy classifier weights tại models/stage2_best.pth, sử dụng weights khởi tạo!
[INFO] Bắt đầu đánh giá trên 225 ảnh test...
Evaluating: 100%|█████████████████████████████| 225/225 [00:42<00:00,  5.35it/s]

  KẾT QUẢ ĐÁNH GIÁ (PER CLASS)
            Precision  Recall      F1
Background     0.0000  0.0000  0.0000
Glass          0.0000  0.0000  0.0000
Metal          0.0203  0.3333  0.0383
Other          0.0130  0.0976  0.0230
Paper          0.0115  0.2105  0.0217
Plastic        0.0000  0.0000  0.0000
--------------------------------------------------
  Macro Precision : 0.0075
  Macro Recall    : 0.1069
  Macro F1 (≈mAP) : 0.0138

[INFO] Đã lưu kết quả tại: results/2stage_eval

✅ ĐÁNH GIÁ HOÀN TẤT! Kết quả được lưu tại results/2stage_eval


In [7]:
# ============================================================
# 7. Zip tất cả kết quả để tải về
# ============================================================
import shutil

# Nén thư mục kết quả
shutil.make_archive("/kaggle/working/final_results", "zip", "/kaggle/working/waste-detection2-Stage/results")
print("\n📦 Đã nén toàn bộ logs, biểu đồ và kết quả: /kaggle/working/final_results.zip")

# Nén weights của model
if os.path.exists("/kaggle/working/waste-detection2-Stage/models"):
    shutil.make_archive("/kaggle/working/final_models", "zip", "/kaggle/working/waste-detection2-Stage/models")
    print("📦 Đã nén trọng số mô hình: /kaggle/working/final_models.zip")


📦 Đã nén toàn bộ logs, biểu đồ và kết quả: /kaggle/working/final_results.zip
